# Notebook to see some cutouts in NISP vs. NIRCAM

In [ ]:
import dropbox
import io
import json
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from astropy.wcs import WCS
from astropy.nddata import Cutout2D

# in HUDF:
jwst_url = 'https://www.dropbox.com/scl/fo/s5e16qezp24g8fuxwn9zo/AFpIqjZNpApPmAyJTBeq_cE?rlkey=twb98st99w1hagqp95a9qlces&dl=0'
nisp_url = 'https://www.dropbox.com/scl/fo/mhtu22pidzezlw4vh035s/ADERx2CNIhGoNftvmw7xMHY?rlkey=dc9gq4jpqxi49hvijalgq53dz&dl=0'
cat_path = '../catalog/gds.fits'

# in cosmos 
jwst_url = 'https://www.dropbox.com/scl/fo/adjat4nzyut3zijz8b6i3/ACOIMQ2yHcRoTdkfayMh0XE?rlkey=qamhxolhsmz05hs747aiiencm&dl=0'
nisp_url = 'https://www.dropbox.com/scl/fo/kej9smd6kjfqd3e1xj3pn/AN-77wdF13ups2AFvNuCpsw?rlkey=sludiaw7bb7fn82q20p9glwr2&dl=0'
cat_path = '../catalog/COSMOS2020_CLASSIC_R1_v2.2_p3.fits'

# Initialize Dropbox
home = "/Users/shemmati"
with open(f"{home}/secrets/dropbox_token") as token_file:
    token = token_file.read().strip()

dbx = dropbox.Dropbox(token)

# Load FITS footprint indices
with open("older/jwst_index.json", "r") as f:
    jwst_fits_index = json.load(f)
with open("older/nisp_index.json", "r") as f:
    nisp_fits_index = json.load(f)


In [ ]:
# Function to find matching FITS files
def find_matching_fits(ra, dec, fits_index):
    return [
        entry["file_name"] for entry in fits_index
        if entry["ra_min"] <= ra <= entry["ra_max"] and entry["dec_min"] <= dec <= entry["dec_max"]
    ]
    
# Load the FITS file
with fits.open(cat_path) as hdul:
    cat_data = hdul[1].data  # Adjust the HDU index if your data is not in the first extension


# Step 1: Pre-filter galaxies that are in both JWST & Euclid
sel = (cat_data['lp_zBEST'] > 0.01) & (cat_data['lp_zBEST'] < 1) & (cat_data['ACS_F814W_MAG'] < 23)& (cat_data['KRON_RADIUS'] >0.01)
filtered = cat_data[sel]

# Precompute which galaxies are inside both JWST & Euclid FITS files
candidates = []
for i in range(len(filtered)):
    ra, dec = filtered['ALPHA_J2000'][i], filtered['DELTA_J2000'][i]
    jwst_files = find_matching_fits(ra, dec, jwst_fits_index)
    nisp_files = find_matching_fits(ra, dec, nisp_fits_index)

    if jwst_files and nisp_files:
        candidates.append((ra, dec, jwst_files[0], nisp_files[0]))  # Select first available file


In [ ]:
len(candidates)

In [ ]:
candidates[0]

In [ ]:
selected_galaxies = []
ra_selected, dec_selected, file_j, file_n=candidates[0] # Limit to 3 galaxies
def get_cutout_from_fits(dbx, shared_url, file_name, ra, dec, size=50):
    """Extract cutout from a FITS file."""
    _, res = dbx.sharing_get_shared_link_file(url=shared_url, path=f"/{file_name}")
    fits_data = io.BytesIO(res.content)

    with fits.open(fits_data, memmap=True) as hdul:
        wcs = WCS(hdul[0].header)
        x, y = wcs.all_world2pix(ra, dec, 1)

        return Cutout2D(hdul[0].data, (x, y), size, wcs=wcs)


cutout_j = get_cutout_from_fits(dbx, jwst_url, file_j, ra_selected, dec_selected, size=50)
cutout_n = get_cutout_from_fits(dbx, nisp_url, file_n, ra_selected, dec_selected, size=50)

if cutout_j and cutout_n:
    selected_galaxies.append((ra_selected, dec_selected, cutout_j, cutout_n))

In [ ]:
# Function to find matching FITS files
def find_matching_fits(ra, dec, fits_index):
    """Finds FITS files that may contain the given RA/DEC using precomputed bounding boxes."""
    return [
        entry["file_name"] for entry in fits_index
        if entry["ra_min"] <= ra <= entry["ra_max"] and entry["dec_min"] <= dec <= entry["dec_max"]
    ]

# Function to get cutouts
def get_cutout_from_fits(dbx, shared_url, fits_index, ra, dec, size=50):
    """Finds the first matching FITS file and extracts a cutout."""
    possible_files = find_matching_fits(ra, dec, fits_index)

    if not possible_files:
        print("No matching FITS files found in index.")
        return None, None

    for file_name in possible_files:
        print(f"Checking {file_name}...")

        try:
            _, res = dbx.sharing_get_shared_link_file(url=shared_url, path=f"/{file_name}")
            fits_data = io.BytesIO(res.content)

            with fits.open(fits_data, memmap=True) as hdul:
                wcs = WCS(hdul[0].header)
                x, y = wcs.all_world2pix(ra, dec, 1)

                cutout = Cutout2D(hdul[0].data, (x, y), size, wcs=wcs)
                return cutout, file_name  # Stop immediately after finding a match

        except Exception as e:
            print(f"Error processing {file_name}: {e}")

    print("RA/DEC does not fall in any available FITS file.")
    return None, None

# Example usage
cutout_n, matching_file_n = get_cutout_from_fits(dbx, "nisp-shared-url", nisp_fits_index, ra_selected, dec_selected, size=50) # 300 mas?
cutout_j, matching_file_j = get_cutout_from_fits(dbx, "jwst-shared-url", jwst_fits_index, ra_selected, dec_selected, size=50) # 60 mas?


In [ ]:
# Step 2: Process only the first N valid galaxies
selected_galaxies = []
for ra_selected, dec_selected, file_j, file_n in candidates[:3]:  # Limit to 3 galaxies
    def get_cutout_from_fits(dbx, shared_url, file_name, ra, dec, size=50):
        """Extract cutout from a FITS file."""
        _, res = dbx.sharing_get_shared_link_file(url=shared_url, path=f"/{file_name}")
        fits_data = io.BytesIO(res.content)

        with fits.open(fits_data, memmap=True) as hdul:
            wcs = WCS(hdul[0].header)
            x, y = wcs.all_world2pix(ra, dec, 1)

            return Cutout2D(hdul[0].data, (x, y), size, wcs=wcs)


    cutout_j = get_cutout_from_fits(dbx, jwst_url, file_j, ra_selected, dec_selected, size=50)
    cutout_n = get_cutout_from_fits(dbx, nisp_url, file_n, ra_selected, dec_selected, size=50)

    if cutout_j and cutout_n:
        selected_galaxies.append((ra_selected, dec_selected, cutout_j, cutout_n))

# Step 3: Plot the results
for idx, (ra, dec, cutout_j, cutout_n) in enumerate(selected_galaxies):
    plt.figure(figsize=(10, 5))

    plt.subplot(1, 2, 1)
    plt.imshow(cutout_n.data, origin="lower", cmap="gray")
    plt.title(f"Euclid NISP @ ({ra:.5f}, {dec:.5f})")

    plt.subplot(1, 2, 2)
    plt.imshow(cutout_j.data, origin="lower", cmap="gray")
    plt.title(f"JWST NIRCam @ ({ra:.5f}, {dec:.5f})")

    plt.show()


In [ ]:
selected_galaxies